# S14 Grid to Map｜Jackson Heights + Corona 学习版
根据 [Ivory Wang 的 S14](https://github.com/ivorywang-1005/intro-to-urban-data/blob/main/Scale/S14_Grid_to_Map.ipynb) 改写。按原练习的四步运行，每一步查看变量和预览。最后在 Rhino 打开导出的 `grid_to_map.3dm`，再用老师的 [S15 定义](https://github.com/ivorywang-1005/intro-to-urban-data/blob/main/Scale/S15_Grid_To_Map.ghx) 读 `lots_grid.csv`。课程的 S15 用原生组件连接两组点。

| S14 原变量 | 这里代表什么 |
|---|---|
| `manhattan_geom` | Jackson Heights、North Corona、Corona 的研究边界 |
| `lots_gdf` | 已有分析字段的地块 |
| `lots_stats_filtered` | 五层筛选后的空地与地面停车场 |
| `lots_map` | 裁切后位于地图原位的轮廓 |
| `lots_grid` | 同一批入选地块在地图上方的副本 |


## 0｜Setup
挂载你保存了分析成果的 Google Drive。设置 `COLS` 可以调整每行的卡片数量。

In [ ]:
%pip -q install rhino3dm geopandas matplotlib
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime
from pathlib import Path
import json
import math
import shutil
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import pandas as pd
import rhino3dm
from shapely.affinity import translate, scale
from IPython.display import display

ROOT = Path("/content/drive/MyDrive/ARCH6131_Scale")
RAW = ROOT / "data/raw"
OUT = ROOT / "outputs"
CRS = 2263
COLS = 4


## 1｜Fetch data → 换成自己已保存的数据
S14 读取行政区、MapPLUTO 和建筑轮廓。你已经完成这些数据的整理，因此这里读取 `boundary.geojson` 与 `all_enriched_plots.geojson`。输出是统一到 EPSG:2263 的地块表及地块编号。

In [ ]:
boundary_gdf = gpd.read_file(RAW / "boundary.geojson").to_crs(CRS)
study_area = boundary_gdf.geometry.union_all()
lots_gdf = gpd.read_file(OUT / "all_enriched_plots.geojson").to_crs(CRS)

# BBL 是 S15 的配对键。把已经验证过的税务地块 plot_id 作为 BBL。
lots_gdf["BBL"] = lots_gdf["plot_id"].astype("string").str.strip()
if not lots_gdf["BBL"].str.fullmatch(r"\d{10}", na=False).all():
    raise ValueError("有地块 ID 不是十位税务地块编号，请先检查")
if not lots_gdf["BBL"].is_unique:
    raise ValueError("地块编号重复")
print("地块数量", len(lots_gdf), "｜坐标系", lots_gdf.crs)
display(lots_gdf[["BBL", "LandUse", "BldgClass"]].head())


## 2｜Join and measure → 检查原有指标
S14 用 overlay 和 groupby 算建筑覆盖。这里接用你此前计算的结果。比较 `coverage_percentage` 与 `building_coverage_pct` 的单位：原练习的覆盖率是 0–1，当前数据是 0–100。

In [ ]:
metrics = [
    "building_coverage_pct", "food_store_count_buffer", "crash_count_buffer",
    "ped_casualty_crash_count_buffer", "age65_pct", "adult_asthma_pct"
]
for field in metrics:
    lots_gdf[field] = pd.to_numeric(lots_gdf[field], errors="coerce")
display(lots_gdf[metrics].describe().T)
display(lots_gdf[metrics].isna().sum().rename("缺失地块数"))


## 3｜Filter → 用之前的五层规则筛选
保留 S14 的 `lots_stats_filtered` 变量。依次筛建筑覆盖、统计区年龄、周边食品店、行人伤亡事故、现状用途。阈值仍从初始集合计算。输出每层数量与保留的地块。


In [ ]:
age_by_tract = lots_gdf[["age65_pct_GEOID", "age65_pct"]].dropna().drop_duplicates()
if not age_by_tract.age65_pct_GEOID.is_unique:
    raise ValueError("同一统计区有多个年龄比例，请检查源数据")

coverage_limit = lots_gdf["building_coverage_pct"].quantile(.25)
age_limit = age_by_tract.age65_pct.median()
food_limit = lots_gdf["food_store_count_buffer"].median()
ped_limit = lots_gdf["ped_casualty_crash_count_buffer"].median()
if not all(math.isfinite(v) for v in [coverage_limit, age_limit, food_limit, ped_limit]):
    raise ValueError("筛选阈值缺少数据")

land_use = pd.to_numeric(lots_gdf["LandUse"], errors="coerce")
building_class = lots_gdf["BldgClass"].astype("string").str.strip().str.upper()
vacant = land_use.eq(11)
parking = building_class.isin(["G6", "G7", "Z2", "RP"])
space_type = pd.Series(pd.NA, index=lots_gdf.index, dtype="string")
space_type.loc[vacant] = "Vacant land"
space_type.loc[parking & ~vacant] = "Surface parking"
lots_gdf["space_type"] = space_type

rules = [
    ("Low building coverage", lots_gdf.building_coverage_pct.le(coverage_limit), f"≤ {coverage_limit:.3f}%"),
    ("Higher age 65+ share", lots_gdf.age65_pct.ge(age_limit), f"≥ {age_limit:.3f}%"),
    ("Fewer food retailers", lots_gdf.food_store_count_buffer.le(food_limit), f"≤ {food_limit:g}"),
    ("More pedestrian casualty crashes", lots_gdf.ped_casualty_crash_count_buffer.ge(ped_limit), f"≥ {ped_limit:g}"),
    ("Vacant land or surface parking", vacant | parking, "PLUTO use codes"),
]
keep = pd.Series(True, index=lots_gdf.index)
counts = [("Baseline", "", len(lots_gdf))]
for name, passes, threshold in rules:
    keep &= passes.fillna(False)
    counts.append((name, threshold, int(keep.sum())))
display(pd.DataFrame(counts, columns=["Step", "Threshold", "Retained"]))

lots_stats_filtered = lots_gdf.loc[keep].copy().sort_values("BBL")
lots_stats_filtered["item_id"] = [f"P{i:02d}" for i in range(1, len(lots_stats_filtered)+1)]
if lots_stats_filtered.empty:
    raise ValueError("五层筛选后没有地块，请检查上表")
display(lots_stats_filtered[["item_id", "BBL", "space_type", *metrics]].head(20))


## 4A｜Map → 裁切显示轮廓，再定地图点
将地块与研究范围求交。地图上的所有轮廓都裁切，筛选指标沿用第 3 步。为了让 Rhino 模型靠近原点，两组几何共同减去同一个 EPSG:2263 原点。每个入选地块保留一个内部点作为 Map Anchor。

In [ ]:
lots_map = gpd.clip(lots_gdf, study_area, keep_geom_type=True)
lots_map = lots_map.loc[
    lots_map.geom_type.isin(["Polygon", "MultiPolygon"]) & lots_map.area.gt(0)
].copy()
if lots_map.empty:
    raise ValueError("没有地块在研究边界内")

origin_x, origin_y, _, _ = study_area.bounds
lots_map.geometry = lots_map.geometry.translate(xoff=-origin_x, yoff=-origin_y)
local_boundary = translate(study_area, xoff=-origin_x, yoff=-origin_y)

selected_meta = lots_stats_filtered[["BBL", "item_id"]]
map_selected = lots_map.merge(selected_meta, on="BBL", validate="one_to_one")
if map_selected.empty:
    raise ValueError("裁切后没有可绘制的入选地块")
map_selected["map_anchor"] = map_selected.geometry.representative_point()
# 保留 S14 的从北到南排序，同时保持 BBL 和 item_id 恒定。
map_selected["cy"] = map_selected.map_anchor.y
map_selected = map_selected.sort_values(["cy", "BBL"], ascending=[False, True]).reset_index(drop=True)
print("地图轮廓地块", len(lots_map), "｜实际绘制卡片", len(map_selected))
display(map_selected[["item_id", "BBL", "map_anchor"]].head())


## 4B｜Grid → 用 divmod 分行，translate 移到地图上方
S14 用 `row, col = divmod(i, COLS)` 得到行列，再把地块移到单元格中心。这里沿用同样的算法，行从上向下排列。所有卡片用相同的比例，改变 `CARD_SCALE` 可以比较布局。点和曲线使用同一次缩放及平移，才能保持配对。

In [ ]:
CARD_SCALE = 1.0
if COLS < 1 or not math.isfinite(CARD_SCALE) or CARD_SCALE <= 0:
    raise ValueError("列数与卡片比例需为正数")

bounds = map_selected.bounds
card_span = max((bounds.maxx - bounds.minx).max(), (bounds.maxy - bounds.miny).max())
map_xmin, _, map_xmax, map_ymax = local_boundary.bounds
cell = max(card_span * CARD_SCALE * 1.4, (map_xmax - map_xmin) / COLS)
rows = math.ceil(len(map_selected) / COLS)
gap = cell * .6
grid_geoms, grid_points = [], []

for i, row_data in map_selected.iterrows():
    row, col = divmod(i, COLS)
    source_anchor = row_data.map_anchor
    cx = map_xmin + (col + .5) * cell
    cy = map_ymax + gap + (rows - row - .5) * cell

    # 以同一个地块点为原点缩放，再把该点移到格子中心。
    scaled = scale(row_data.geometry, xfact=CARD_SCALE, yfact=CARD_SCALE,
                   origin=(source_anchor.x, source_anchor.y))
    dx, dy = cx - source_anchor.x, cy - source_anchor.y
    grid_geoms.append(translate(scaled, xoff=dx, yoff=dy))
    grid_points.append(translate(source_anchor, xoff=dx, yoff=dy))

lots_grid = map_selected.copy()
lots_grid.geometry = grid_geoms
lots_grid["grid_anchor"] = grid_points
assert lots_grid["BBL"].tolist() == map_selected["BBL"].tolist()
display(lots_grid[["item_id", "BBL", "map_anchor", "grid_anchor"]].head())


## 4C｜在 Colab 检查一对点如何连成线
这里的线仅供学习预览。Rhino 文件输出曲线和锚点，正式连线由 S15 根据 BBL 生成。

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))
lots_map.boundary.plot(ax=ax, color="#6784a4", linewidth=.22)
map_selected.boundary.plot(ax=ax, color="#c14f30", linewidth=1)
lots_grid.boundary.plot(ax=ax, color="#c14f30", linewidth=1)
links = [
    [(a.x, a.y), (b.x, b.y)]
    for a, b in zip(lots_grid.map_anchor, lots_grid.grid_anchor)
]
ax.add_collection(LineCollection(links, colors="#9aaab2", linewidths=.45))
for item_id, pt in zip(lots_grid.item_id, lots_grid.grid_anchor):
    ax.annotate(item_id, (pt.x, pt.y), xytext=(4, 4),
                textcoords="offset points", fontsize=8)
ax.set_title("Grid to map | Jackson Heights and Corona")
ax.set_aspect("equal")
ax.set_axis_off()
plt.show()


## 5A｜S14 的 Rhino 图层与转换函数
看 `to_rhino_curves()`。它将每个多边形环的 XY 坐标写成一条闭合 PolylineCurve。多部件地块可以有多条轮廓。每个 BBL 在 Map Anchor 和 Grid Anchor 两个图层里各有一个点。

In [ ]:
model = rhino3dm.File3dm()
model.Settings.ModelUnitSystem = rhino3dm.UnitSystem.Feet

def add_layer(name, rgb):
    layer = rhino3dm.Layer()
    layer.Name = name
    layer.Color = (*rgb, 255)
    index = model.Layers.Add(layer)
    if index < 0:
        raise RuntimeError(f"无法创建图层 {name}")
    return index

layers = {
    "grid_geometry": add_layer("Grid Geometry", (186, 77, 47)),
    "grid_anchor": add_layer("Grid Anchor", (222, 98, 62)),
    "map_geometry": add_layer("Map Geometry", (74, 105, 149)),
    "map_anchor": add_layer("Map Anchor", (17, 113, 99)),
    "sites": add_layer("Selected Map Geometry", (186, 77, 47)),
    "boundary": add_layer("Study Boundary", (31, 53, 63)),
}

def attrs_for(layer_key, bbl, item_id=None):
    attrs = rhino3dm.ObjectAttributes()
    attrs.LayerIndex = layers[layer_key]
    attrs.Name = str(bbl)  # S15 的 Model Object → Name 用这个值
    if item_id is not None:
        attrs.SetUserString("item_id", str(item_id))
    return attrs

def to_rhino_curves(geometry):
    if geometry.geom_type == "Polygon":
        for ring in [geometry.exterior, *geometry.interiors]:
            polyline = rhino3dm.Polyline()
            for coordinate in ring.coords:
                polyline.Add(coordinate[0], coordinate[1], 0)
            yield polyline.ToPolylineCurve()
    elif hasattr(geometry, "geoms"):
        for part in geometry.geoms:
            yield from to_rhino_curves(part)

def add_curves(geometry, layer_key, bbl, item_id=None):
    for curve in to_rhino_curves(geometry):
        model.Objects.AddCurve(curve, attrs_for(layer_key, bbl, item_id))

def add_anchor(point, layer_key, bbl, item_id):
    model.Objects.AddPoint(
        rhino3dm.Point3d(point.x, point.y, 0),
        attrs_for(layer_key, bbl, item_id)
    )


## 5B｜写出 .3dm 和 CSV
`lots_grid.csv` 保留 S15 使用的五列结构：BBL、食品店数、行人伤亡事故数、建筑覆盖率、Item ID。S15 前四个文字标签可用前四列。完整指标保存在 `candidate_items.csv`，其中包括年龄与健康。

In [ ]:
# 重跑本格前先重新运行 5A，建立空模型。
if len(model.Objects):
    raise ValueError("请先重新运行 5A 建立空模型，再执行本格")
add_curves(local_boundary, "boundary", "Study boundary")

for row in lots_map.itertuples():
    add_curves(row.geometry, "map_geometry", row.BBL)

for row in map_selected.itertuples():
    add_curves(row.geometry, "sites", row.BBL, row.item_id)
    add_anchor(row.map_anchor, "map_anchor", row.BBL, row.item_id)

for row in lots_grid.itertuples():
    add_curves(row.geometry, "grid_geometry", row.BBL, row.item_id)
    add_anchor(row.grid_anchor, "grid_anchor", row.BBL, row.item_id)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
folder = OUT / ("S14_Queens_Grid_to_Map_" + stamp)
folder.mkdir(parents=True)
rhino_path = folder / "grid_to_map.3dm"
if not model.Write(str(rhino_path), 7):
    raise IOError("Rhino 文件写入失败")

s15_columns = ["BBL", "food_store_count_buffer",
               "ped_casualty_crash_count_buffer", "building_coverage_pct", "item_id"]
lots_grid[s15_columns].round(2).to_csv(folder / "lots_grid.csv", index=False)
lots_grid[["BBL", "item_id", "space_type", *metrics]].to_csv(
    folder / "candidate_items.csv", index=False
)
(folder / "coordinate_origin.json").write_text(
    json.dumps({"crs": "EPSG:2263", "origin_xy": [origin_x, origin_y],
                "units": "US survey feet", "card_scale": CARD_SCALE}, indent=2),
    encoding="utf-8"
)

# 重新读文件，确认两组点的名称都与 CSV 的 BBL 对应。
check_model = rhino3dm.File3dm.Read(str(rhino_path))
if check_model is None:
    raise IOError("导出的 Rhino 文件无法重新读取")
csv_bbl = pd.read_csv(folder / "lots_grid.csv", dtype={"BBL": str}).BBL.tolist()
for key in ["map_anchor", "grid_anchor"]:
    names = [
        obj.Attributes.Name for obj in check_model.Objects
        if obj.Attributes.LayerIndex == layers[key]
    ]
    if len(names) != len(csv_bbl) or set(names) != set(csv_bbl):
        raise ValueError(f"{key} 的名称与 CSV 不一致")
archive = shutil.make_archive(str(folder), "zip", root_dir=folder)
print(f"已导出 {len(csv_bbl)} 对锚点，下载并解压：{archive}")
display(lots_grid[s15_columns].head())


## 6｜在 S15 里读文件、看配对
1. 在 Rhino 直接 **Open** 新的 `grid_to_map.3dm`，Top 视图里 Zoom Extents。图层面板能看到两组几何和两组点。
2. 打开课程的 [S15_Grid_To_Map.ghx](https://github.com/ivorywang-1005/intro-to-urban-data/blob/main/Scale/S15_Grid_To_Map.ghx)。在左上 **File Path** 参数重新设置解压目录里的 `lots_grid.csv`。
3. 看 **Query Model Objects → Model Object → Name**。Grid Anchor 与 Map Anchor 的 Name 都是 BBL。原版 **Member Index** 用 CSV 的 BBL 找到地图点的下标，**List Item** 取出对应地图点，**Line** 连接到网格点。
4. 看 **Read File → Split List → Text Split → Flip Matrix → Explode Tree**。第 `{0}` 列是 BBL，`{1}` 食品店数，`{2}` 行人伤亡事故数，`{3}` 建筑覆盖率（%），`{4}` 项目编号。把老师的三个文字前缀改为 `Food retailers:`、`Pedestrian casualty crashes:`、`Building coverage (%):`。
5. 核对同一张卡片的 BBL、连线和 CSV 行。原版连线和文字都依赖 Grid Anchor 的顺序与 CSV 一致。给 Grid Anchor 的 Query 输出接一个 Model Object，再把 Name 接 Panel，与 CSV `{0}` 对照。若顺序不同，增加一组 Member Index（S 接 Grid Anchor 名称，M 接 CSV BBL）和 List Item（L 接 Grid Anchor 几何，i 接匹配下标），将重新排列的点同时接到 Line 的网格端和四个 Move 的几何端。
6. 保存自己的 Rhino 文件与 Grasshopper 定义。想增加年龄或哮喘标签，可以参照上一条连接方式读取完整指标 CSV，再按 BBL 取值。
